# Conformal prediction

Paper-facing tables for the 90% conformal prediction experiments. The analysis uses the linear probe, excludes TENX143 and Hibou-L, retains only LEMON ViT-S/8 under the label LEMON, and labels the four-layer, all-training-cells, adapted CellBridge representation simply as CellBridge.

The table compares representations under branch graph, the proposed conformal construction. HEST (ID) and all OOD benchmarks appear in the same table. The CovGap ablation over conformal constructions is in notebook 10.


In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

WORKSPACE = Path(os.environ.get(
    "CELLBRIDGE_WORKSPACE",
    "/lustre/scratch126/cellgen/lotfollahi/so25",
))
RESULTS_ROOT = Path(os.environ.get(
    "CELLBRIDGE_INFERENCE_RESULTS",
    WORKSPACE / "outputs/reproducibility/inference",
))
CELLBRIDGE_RESULTS_ROOT = Path(os.environ.get(
    "CELLBRIDGE_4L_INFERENCE_RESULTS",
    WORKSPACE / "outputs/reproducibility/ablations/lora_depth/layers_4_cells_all/inference",
))

PROBE = "linear"
EXCLUDED_HEST_SAMPLES = {"TENX143"}
METHOD_COMPARISON_CP = "branch_graph"

METHOD_GROUPS = [
    ("Patch-level FMs", ["UNI2-h", "Virchow2"]),
    ("Cell-level FMs", ["VOLTA", "LEMON"]),
    ("Multi-modal FMs", ["PAST", "SpatialWhisperer"]),
    ("Ours", ["CellBridge"]),
]
METHOD_ORDER = [
    method
    for _, methods in METHOD_GROUPS
    for method in methods
]

BENCHMARK_ORDER = [
    "HEST",
    "NuCLS",
    "PanNuke",
    "Lizard",
    "PhenoCell (all)",
    "PhenoCell (immune)",
]
BENCHMARK_LABELS = {
    "HEST": "HEST (ID)",
    "Lizard": "Lizard (OOD)",
    "NuCLS": "NuCLS (OOD)",
    "PanNuke": "PanNuke (OOD)",
    "PhenoCell (all)": "PhenoCell all (OOD)",
    "PhenoCell (immune)": "PhenoCell immune (OOD)",
}

TABLE_METRICS = [
    ("mean_set_size", "Mean set size (↓)"),
    ("singleton_rate", "Singleton rate (↑)"),
]
COVERAGE_METRICS = [("marginal_coverage", "Marginal coverage (%)")]
LATEX_METRIC_SECTIONS = {
    "marginal_coverage": r"Marginal Coverage (\%)",
    "mean_set_size": r"Mean Set Size ($\downarrow$)",
    "singleton_rate": r"Singleton Rate ($\uparrow$)",
}
LATEX_METHOD_LABELS = {"SpatialWhisperer": "SW"}

ROW_COLOUR = "gray!15"
OURS_BACKGROUND = "#E6E6E6"

## Load and aggregate the conformal metrics

HEST entries are means across the 14 retained tissue sections; each OOD entry comes from its fixed held-out test split. Singleton rate is reported as a percentage. Mean set size is the mean number of labels retained per prediction set.


In [ ]:
BASELINE_LABELS = {
    "uni": "UNI2-h",
    "virchow2": "Virchow2",
    "lemon": "LEMON",
    "past": "PAST",
    "volta": "VOLTA",
    "spatialwhisperer": "SpatialWhisperer",
}


def paper_method(method, version, arm):
    if method == "cellbridge" or arm.startswith("lora_"):
        size = version.removeprefix("cells_")
        size = size.upper() if size != "all" else "all"
        representation = arm.removeprefix("lora_")
        if size == "all" and representation == "adapted":
            return "CellBridge"
        return "CellBridge {} ({})".format(size, representation)
    return BASELINE_LABELS.get(method, method)


def benchmark_label(dataset, task):
    if dataset == "phenocell":
        return "PhenoCell ({})".format(task)
    return {
        "hest": "HEST",
        "lizard": "Lizard",
        "nucls": "NuCLS",
        "pannuke": "PanNuke",
    }[dataset]


def load_conformal_metrics(root, cellbridge_root):
    if not cellbridge_root.is_dir():
        raise FileNotFoundError(
            "Four-layer CellBridge results do not exist: {}".format(cellbridge_root)
        )
    paths = list(root.glob(
        "*/*/*/evaluation/*/*/*/conformal_prediction/metrics.csv"
    ))
    paths += list(root.glob(
        "*/*/hest/*/evaluation/*/*/*/conformal_prediction/metrics.csv"
    ))
    # The four-layer run has dataset directories directly below its root.
    paths += list(cellbridge_root.glob(
        "*/evaluation/*/*/*/conformal_prediction/metrics.csv"
    ))
    paths += list(cellbridge_root.glob(
        "hest/*/evaluation/*/*/*/conformal_prediction/metrics.csv"
    ))
    tables = []
    replaced_results, replacement_results = set(), set()
    required = {
        "method",
        "arm",
        "task",
        "probe",
        "conformal_method",
        "target_coverage",
        "marginal_coverage",
        "mean_prediction_set_size",
        "singleton_rate",
    }
    for path in sorted(paths):
        from_four_layer = path.is_relative_to(cellbridge_root)
        if from_four_layer:
            # Prefix the implied method/version so both layouts parse alike.
            parts = ("cellbridge", "cells_all") + path.relative_to(cellbridge_root).parts
        else:
            parts = path.relative_to(root).parts
        method, version, dataset = parts[:3]
        if dataset == "hest":
            sample_id = parts[3]
            arm, task, probe = parts[5:8]
        else:
            sample_id = pd.NA
            arm, task, probe = parts[4:7]

        # CellBridge all/adapted comes only from the four-layer run; the
        # two-layer cells_all/lora_adapted results are replaced, never a fallback.
        result_key = (dataset, str(sample_id), task, probe)
        if from_four_layer:
            if arm != "lora_adapted":
                continue
            replacement_results.add(result_key)
        elif method == "cellbridge" and version == "cells_all" and arm == "lora_adapted":
            if str(sample_id) not in EXCLUDED_HEST_SAMPLES:
                replaced_results.add(result_key)
            continue

        table = pd.read_csv(path)
        missing = required.difference(table.columns)
        if missing:
            raise ValueError(
                "{} is missing columns: {}".format(path, sorted(missing))
            )
        table["version"] = version
        table["sample_id"] = sample_id
        table["paper_method"] = paper_method(method, version, arm)
        table["benchmark"] = benchmark_label(dataset, task)
        table["result_dir"] = str(path.parent)
        tables.append(table)

    missing_replacements = replaced_results - replacement_results
    if missing_replacements:
        raise FileNotFoundError(
            "Four-layer CellBridge results are missing for: {}".format(
                sorted(missing_replacements)
            )
        )
    if not tables:
        raise FileNotFoundError(
            "No conformal metrics found below {}".format(root)
        )

    table = pd.concat(tables, ignore_index=True)
    table = table.loc[table["method"] != "hibou_l"]
    table = table.loc[
        ~((table["method"] == "lemon") & (table["version"] != "vits8"))
    ]
    table = table.loc[
        ~table["sample_id"].astype(str).isin(EXCLUDED_HEST_SAMPLES)
    ]
    return table.reset_index(drop=True)


def aggregate_points(table):
    return (
        table.groupby(
            ["paper_method", "benchmark", "probe", "conformal_method"],
            as_index=False,
        )
        .agg(
            marginal_coverage=("marginal_coverage", "mean"),
            mean_set_size=("mean_prediction_set_size", "mean"),
            singleton_rate=("singleton_rate", "mean"),
            target_coverage=("target_coverage", "mean"),
            n_hest_sections=("sample_id", "count"),
        )
    )


conformal = load_conformal_metrics(RESULTS_ROOT, CELLBRIDGE_RESULTS_ROOT)
points = aggregate_points(conformal)

target_coverages = conformal["target_coverage"].dropna().unique()
if len(target_coverages) != 1:
    raise ValueError(
        "Expected one nominal coverage target, found {}".format(
            sorted(target_coverages)
        )
    )
NOMINAL_COVERAGE = float(target_coverages[0])

print(
    "Loaded {:,} conformal rows at {:.0%} nominal coverage; "
    "probe = {}; excluded HEST = {}".format(
        len(conformal),
        NOMINAL_COVERAGE,
        PROBE,
        sorted(EXCLUDED_HEST_SAMPLES),
    )
)
print(
    "CellBridge = four-layer all/adapted results from {}".format(
        CELLBRIDGE_RESULTS_ROOT
    )
)

## Representation comparison under branch graph

This table compares all baseline representations with CellBridge using branch graph and the linear probe. Methods are grouped by representation family. The displayed table stacks the two metrics beneath each method; the LaTeX table is transposed like notebook 07's, with methods across the top and one block of dataset rows per metric (mean set size, then singleton rate), each split into rotated ID and OOD groups. The class-conditional coverage gap (CovGap) is in notebook 10. In the displayed table, dataset columns are grouped into ID and OOD. Gray shading on CellBridge identifies our proposed representation and conformal construction.

Every entry carries an error. HEST entries report the mean ± standard error across the 14 retained sections: their standard deviation divided by √14 (notebook 07 reports the standard deviation instead). Each OOD entry reports the grouped-bootstrap standard error stored in the run's `conformal_prediction/uncertainty.csv`. The pipeline writes it with the macro F1 bootstrap settings: the same test groups (slides for NuCLS, samples for Lizard and PanNuke, regions for PhenoCell), 1,000 resamples, and seed. The calibration threshold stays fixed, so these errors reflect test-set sampling, not calibration variability.


In [ ]:
def _format_metric(value, metric):
    if pd.isna(value):
        return "--"
    if metric in {"singleton_rate", "marginal_coverage"}:
        return "{:.1f}".format(100 * value)
    return "{:.2f}".format(value)


TABLE_BENCHMARK_LABELS = {
    "HEST": "HEST",
    "Lizard": "Lizard",
    "NuCLS": "NuCLS",
    "PanNuke": "PanNuke",
    "PhenoCell (all)": "PhenoCell all",
    "PhenoCell (immune)": "PhenoCell immune",
}
LATEX_BENCHMARK_LABELS = {
    "PhenoCell all": r"PhenoCell\textsubscript{all}",
    "PhenoCell immune": r"PhenoCell\textsubscript{imm}",
}
BENCHMARK_SPLITS = {
    benchmark: "ID" if benchmark == "HEST" else "OOD"
    for benchmark in BENCHMARK_ORDER
}


def build_rowwise_table(
    rows,
    row_key,
    row_order,
    row_heading,
    row_labels=None,
    metrics=None,
):
    metrics = TABLE_METRICS if metrics is None else metrics
    duplicate = rows.duplicated([row_key, "benchmark"], keep=False)
    if duplicate.any():
        raise ValueError(
            "Duplicate table entries:\n{}".format(
                rows.loc[
                    duplicate,
                    [row_key, "benchmark"],
                ].to_string(index=False)
            )
        )

    observed = set(
        rows[[row_key, "benchmark"]].itertuples(index=False, name=None)
    )
    missing = [
        (row, benchmark)
        for row in row_order
        for benchmark in BENCHMARK_ORDER
        if (row, benchmark) not in observed
    ]
    if missing:
        raise ValueError("Missing table entries: {}".format(missing))

    metric_keys = [metric for metric, _ in metrics]
    raw = rows.pivot(
        index=row_key,
        columns="benchmark",
        values=metric_keys,
    )
    labels = row_labels or {}
    values = []
    index = []
    for row in row_order:
        row_label = labels.get(row, row)
        for metric, metric_label in metrics:
            index.append((row_label, metric_label))
            values.append([
                _format_metric(raw.loc[row, (metric, benchmark)], metric)
                for benchmark in BENCHMARK_ORDER
            ])

    columns = pd.MultiIndex.from_tuples(
        [
            (
                BENCHMARK_SPLITS[benchmark],
                TABLE_BENCHMARK_LABELS[benchmark],
            )
            for benchmark in BENCHMARK_ORDER
        ],
        names=["Split", "Dataset"],
    )
    return pd.DataFrame(
        values,
        index=pd.MultiIndex.from_tuples(
            index,
            names=[row_heading, "Metric"],
        ),
        columns=columns,
    )


def rank_table_cells(rows, row_key, row_order, row_labels=None):
    labels = row_labels or {}
    lookup = rows.set_index([row_key, "benchmark"])
    emphasis = {"best": set(), "second": set()}

    for metric, metric_label in TABLE_METRICS:
        for benchmark in BENCHMARK_ORDER:
            candidates = []
            for row in row_order:
                value = lookup.loc[(row, benchmark), metric]
                if pd.isna(value):
                    continue
                displayed_value = float(_format_metric(value, metric))
                if metric == "mean_set_size":
                    score = displayed_value
                else:
                    score = -displayed_value
                candidates.append((row, score))

            score_levels = sorted({score for _, score in candidates})
            for rank, score_level in zip(
                ("best", "second"),
                score_levels[:2],
            ):
                for row, score in candidates:
                    if score == score_level:
                        emphasis[rank].add((
                            labels.get(row, row),
                            metric_label,
                            TABLE_BENCHMARK_LABELS[benchmark],
                        ))
    return emphasis


def display_table(table, emphasis, shaded_rows, label_level):
    def shade(row):
        row_label = row.name[label_level]
        colour = (
            "background-color: {}".format(OURS_BACKGROUND)
            if row_label in shaded_rows
            else ""
        )
        return [colour] * len(row)

    def emphasize(data):
        styles = pd.DataFrame("", index=data.index, columns=data.columns)
        for index in data.index:
            row_label = index[label_level]
            metric_label = index[-1]
            for column in data.columns:
                key = (row_label, metric_label, column[-1])
                if key in emphasis["best"]:
                    styles.at[index, column] = "font-weight: bold"
                elif key in emphasis["second"]:
                    styles.at[index, column] = "text-decoration: underline"
        return styles

    display(
        table.style
        .apply(shade, axis=1)
        .apply(emphasize, axis=None)
        .set_properties(**{"text-align": "center"})
        .set_table_styles([
            {
                "selector": "th.row_heading.level{}".format(label_level),
                "props": [
                    ("text-align", "center"),
                    ("vertical-align", "middle"),
                ],
            }
        ])
    )


def emit_transposed_latex_table(table, caption, label, emphasis, shaded_rows, metrics=None):
    """Print methods across the top and one block of dataset rows per metric,
    with datasets in rotated ID/OOD groups, as in notebook 07."""
    metrics = TABLE_METRICS if metrics is None else metrics
    families = list(dict.fromkeys(index[0] for index in table.index))
    members = {
        family: list(dict.fromkeys(index[1] for index in table.index if index[0] == family))
        for family in families
    }
    methods = [method for family in families for method in members[family]]
    shaded = r">{\columncolor{" + ROW_COLOUR + "}}c"

    def family_spec(family):
        return "".join(shaded if method in shaded_rows else "c" for method in members[family])

    lines = [
        r"\begin{table*}[t]",
        r"\centering",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{cl|" + "|".join(family_spec(family) for family in families) + "}",
        r"\toprule",
    ]
    rules, first = [], 3
    for position, family in enumerate(families):
        count = len(members[family])
        # A single-column group stays a plain cell; \multicolumn would drop its shading.
        heading = r"\textbf{" + family + "}"
        if count > 1:
            align = "c" if position == len(families) - 1 else "c|"
            heading = r"\multicolumn{" + str(count) + "}{" + align + "}{" + heading + "}"
        lines.append((" & & " if position == 0 else "   & ") + heading)
        rules.append(r"\cmidrule(lr){" + "{}-{}".format(first, first + count - 1) + "}")
        first += count
    lines[-1] += r" \\"
    lines += rules
    lines.append(r" & \textbf{Dataset}")
    for family in families:
        lines.append(" & " + " & ".join(
            LATEX_METHOD_LABELS.get(method, method) for method in members[family]
        ))
    lines[-1] += r" \\"

    splits = list(dict.fromkeys(column[0] for column in table.columns))
    for metric, metric_label in metrics:
        lines += [
            r"\midrule",
            r" & & \multicolumn{" + str(len(methods)) + "}{c}{" + LATEX_METRIC_SECTIONS[metric] + r"} \\",
        ]
        for split in splits:
            columns = [column for column in table.columns if column[0] == split]
            rotated = r"\rotatebox[origin=c]{90}{\small\textbf{" + split + "}}"
            lines.append(r"\midrule")
            lines.append(
                rotated if len(columns) == 1
                else r"\multirow{" + str(len(columns)) + "}{*}{" + rotated + "}"
            )
            for column in columns:
                lines.append(" & " + LATEX_BENCHMARK_LABELS.get(column[-1], column[-1]))
                for family in families:
                    cells = []
                    for method in members[family]:
                        value = str(table.loc[(family, method, metric_label), column])
                        key = (method, metric_label, column[-1])
                        if key in emphasis["best"]:
                            value = r"\textbf{" + value + "}"
                        elif key in emphasis["second"]:
                            value = r"\underline{" + value + "}"
                        cells.append(value)
                    lines.append(" & " + " & ".join(cells))
                lines[-1] += r" \\"
    lines += [r"\bottomrule", r"\end{tabular}%", "}", r"\end{table*}"]
    print("\n".join(lines))


main_rows = points.loc[
    (points["probe"] == PROBE)
    & (points["conformal_method"] == METHOD_COMPARISON_CP)
    & points["paper_method"].isin(METHOD_ORDER)
].copy()
main_table = build_rowwise_table(
    main_rows,
    row_key="paper_method",
    row_order=METHOD_ORDER,
    row_heading="Method",
)
family_by_method = {
    method: family
    for family, methods in METHOD_GROUPS
    for method in methods
}
main_table.index = pd.MultiIndex.from_tuples(
    [
        (family_by_method[method], method, metric)
        for method, metric in main_table.index
    ],
    names=["Family", "Method", "Metric"],
)

main_emphasis = rank_table_cells(
    main_rows,
    row_key="paper_method",
    row_order=METHOD_ORDER,
)
main_shaded_rows = {"CellBridge"}


In [ ]:
import numpy as np


def _conformal_uncertainty(result_dir):
    """Bootstrap errors the pipeline wrote next to one OOD run's conformal metrics."""
    path = Path(result_dir) / "uncertainty.csv"
    if not path.is_file():
        raise FileNotFoundError(
            "{} is missing; rerun the conformal evaluation to write it".format(path)
        )
    table = pd.read_csv(path)
    return table.loc[table["conformal_method"] == METHOD_COMPARISON_CP].set_index("metric")


selected = conformal.loc[
    (conformal["probe"] == PROBE)
    & (conformal["conformal_method"] == METHOD_COMPARISON_CP)
    & conformal["paper_method"].isin(METHOD_ORDER)
]
metric_columns = {
    "mean_set_size": "mean_prediction_set_size",
    "singleton_rate": "singleton_rate",
    "marginal_coverage": "marginal_coverage",
}

error_rows = []
for benchmark, rows in selected.groupby("benchmark"):
    if benchmark == "HEST":
        # HEST: standard error of the mean over sections; OOD sets: stored grouped bootstrap.
        for method, sections in rows.groupby("paper_method"):
            for metric, column in metric_columns.items():
                error_rows.append({
                    "paper_method": method, "benchmark": benchmark, "metric": metric,
                    "error": sections[column].std(ddof=1) / np.sqrt(len(sections)),
                    "error_type": "SE over sections",
                    "n_groups": len(sections),
                })
        continue
    for _, row in rows.iterrows():
        uncertainty = _conformal_uncertainty(row["result_dir"])
        for metric, column in metric_columns.items():
            stored = uncertainty.loc[column]
            if not np.isclose(stored["estimate"], row[column]):
                raise ValueError("{}: uncertainty.csv disagrees with metrics.csv".format(row["result_dir"]))
            error_rows.append({
                "paper_method": row["paper_method"], "benchmark": benchmark, "metric": metric,
                "error": stored["bootstrap_se"], "error_type": "bootstrap SE",
                "n_groups": int(stored["n_test_groups"]),
            })
conformal_errors = pd.DataFrame(error_rows)
print(
    conformal_errors.groupby(["benchmark", "error_type"])["n_groups"]
    .agg(["min", "max"]).rename(columns={"min": "groups (min)", "max": "groups (max)"})
    .to_string()
)

error_lookup = conformal_errors.set_index(["paper_method", "benchmark", "metric"])["error"]
metric_of_label = {label: metric for metric, label in TABLE_METRICS + COVERAGE_METRICS}
benchmark_of_label = {TABLE_BENCHMARK_LABELS[benchmark]: benchmark for benchmark in BENCHMARK_ORDER}


def _with_errors(table, template):
    """Copy of the table with each value followed by its error."""
    out = table.copy()
    for row, (_, method, metric_label) in enumerate(table.index):
        metric = metric_of_label[metric_label]
        for position, column in enumerate(table.columns):
            error = error_lookup[(method, benchmark_of_label[column[-1]], metric)]
            shown = "{:.2f}".format(error if metric == "mean_set_size" else 100 * error)
            out.iat[row, position] = template.format(table.iat[row, position], shown)
    return out


display_table(
    _with_errors(main_table, "{} ± {}"),
    emphasis=main_emphasis,
    shaded_rows=main_shaded_rows,
    label_level=1,
)
emit_transposed_latex_table(
    _with_errors(main_table, r"{} \scriptsize{{$\pm$ {}}}"),
    caption=(
        "Representation comparison under branch-conditional graph conformal\n"
        "prediction using linear probes.\n"
        "SS: mean set size ($\\downarrow$); SR: singleton rate (\\%, $\\uparrow$).\n"
        "The nominal coverage target is {0:.0f}\\%. HEST entries report the mean\n"
        "$\\pm$ standard error over 14 held-out sections; OOD entries report\n"
        "grouped-bootstrap standard errors over test slides, samples, or regions,\n"
        "with the calibration threshold held fixed. Bold and underlined values\n"
        "indicate the best and second-best results, respectively. The gray column\n"
        "identifies CellBridge."
    ).format(100 * NOMINAL_COVERAGE),
    label="tab:conformal-representation-comparison",
    emphasis=main_emphasis,
    shaded_rows=main_shaded_rows,
)

## Marginal coverage by representation

Coverage uses the same original `conformal_prediction/metrics.csv` files, selected
`PROBE` and `METHOD_COMPARISON_CP`, method groups, and dataset order as the efficiency
table above. Values are percentages, with the same standard-error conventions:
HEST uses the mean ± SE across the retained sections; OOD uses the stored grouped-bootstrap SE.
The nominal coverage target is printed below. Coverage is a validity criterion,
so the table does not rank higher coverage as better. CellBridge remains shaded.


In [ ]:
coverage_table = build_rowwise_table(
    main_rows,
    row_key="paper_method",
    row_order=METHOD_ORDER,
    row_heading="Method",
    metrics=COVERAGE_METRICS,
)
coverage_table.index = pd.MultiIndex.from_tuples(
    [
        (family_by_method[method], method, metric)
        for method, metric in coverage_table.index
    ],
    names=["Family", "Method", "Metric"],
)
coverage_emphasis = {"best": set(), "second": set()}
print(
    "Marginal coverage (%); nominal target = {:.0%}; probe = {}; conformal method = {}".format(
        NOMINAL_COVERAGE, PROBE, METHOD_COMPARISON_CP
    )
)
display_table(
    _with_errors(coverage_table, "{} ± {}"),
    emphasis=coverage_emphasis,
    shaded_rows=main_shaded_rows,
    label_level=1,
)
emit_transposed_latex_table(
    _with_errors(coverage_table, r"{} \scriptsize{{$\pm$ {}}}"),
    caption=(
        r"Marginal coverage (\%) by representation using "
        + PROBE + " probes and " + METHOD_COMPARISON_CP.replace("_", "-")
        + " conformal prediction. The nominal target is "
        + "{:.0f}".format(100 * NOMINAL_COVERAGE)
        + r"\%. HEST entries report the mean $\pm$ standard error across retained "
        r"sections; OOD entries report grouped-bootstrap standard errors with "
        r"the calibration threshold held fixed. Coverage is not ranked. "
        r"The gray column identifies CellBridge."
    ),
    label="tab:conformal-marginal-coverage",
    emphasis=coverage_emphasis,
    shaded_rows=main_shaded_rows,
    metrics=COVERAGE_METRICS,
)


## Conformal efficiency by dataset

One figure per benchmark task compares the seven representations under the selected
PROBE and METHOD_COMPARISON_CP (linear / branch graph by default). The x-axis is
mean set size and the y-axis is singleton rate. Coverage is not annotated on these figures; the class-conditional coverage
gap (CovGap) is in notebook 10.

Colors, Computer Modern fonts, LaTeX rendering, and axis styling match notebook07.
Figures are narrower and taller (6 by 6.5 inches), with methods listed from top to
bottom in a single-column legend inside the upper-right corner.
HEST points average the 14 retained sections; external points use the fixed test
splits. PhenoCell all and immune are plotted separately. Vector PDFs are saved under
outputs/visualizations/res_figures/conformal_prediction by default.


In [ ]:
import shutil
import subprocess

import matplotlib.pyplot as plt

# Match the TeX setup and plotting conventions in notebook07.
def _ensure_tex():
    missing = [name for name in ("latex", "dvipng") if shutil.which(name) is None]
    if missing:
        modulecmd = shutil.which("modulecmd")
        if modulecmd is None:
            raise RuntimeError("Cannot load texlive: modulecmd is unavailable")
        command = subprocess.run(
            [modulecmd, "python", "load", "texlive-20220321"],
            check=True, capture_output=True, text=True,
        )
        exec(command.stdout, {"os": os})
    missing = [name for name in ("latex", "dvipng") if shutil.which(name) is None]
    if missing:
        raise RuntimeError(
            f"TeX executables remain unavailable: {', '.join(missing)}"
        )

_ensure_tex()
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["cm10"],
    "font.size": 18,
    "axes.edgecolor": "darkgray",
    "axes.grid": True,
    "axes.grid.axis": "y",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.linestyle": "--",
    "grid.color": "lightgray",
    "figure.dpi": 120,
    "savefig.dpi": 300,
})
plt.rc("text.latex", preamble=r"\usepackage{amsmath,bm}")

PLOT_METHOD_LABELS = {"SpatialWhisperer": "SW"}
METHOD_COLORS = {
    "UNI2-h": {"linear": "#CB722F", "knn": "#DEBA9B"},
    "Virchow2": {"linear": "#93300C", "knn": "#C99886"},
    "VOLTA": {"linear": "#BD5673", "knn": "#EAB5C4"},
    "LEMON": {"linear": "#D6A238", "knn": "#EBD19C"},
    "PAST": {"linear": "#8859AB", "knn": "#C0ADD2"},
    "SpatialWhisperer": {"linear": "#287DC5", "knn": "#9CBDDF"},
    "CellBridge": {"linear": "#88C2AB", "knn": "#C9E0D6"},
}

FIGURE_DIR = Path(os.environ.get(
    "CELLBRIDGE_CONFORMAL_FIGURES",
    WORKSPACE / "outputs/visualizations/res_figures/conformal_prediction",
))
SAVE_FIGURES = True
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_BENCHMARKS = {
    "HEST": ("HEST", "hest"),
    "NuCLS": ("NuCLS", "nucls"),
    "PanNuke": ("PanNuke", "pannuke"),
    "Lizard": ("Lizard", "lizard"),
    "PhenoCell (all)": (r"PhenoCell$_{\mathrm{all}}$", "phenocell_all"),
    "PhenoCell (immune)": (r"PhenoCell$_{\mathrm{imm}}$", "phenocell_immune"),
}
scatter_rows = points.loc[
    (points["probe"] == PROBE)
    & (points["conformal_method"] == METHOD_COMPARISON_CP)
    & points["paper_method"].isin(METHOD_ORDER)
].copy()


def plot_conformal_efficiency(benchmark):
    rows = (
        scatter_rows.loc[scatter_rows["benchmark"] == benchmark]
        .set_index("paper_method")
        .loc[METHOD_ORDER]
    )
    title, slug = PLOT_BENCHMARKS[benchmark]
    fig, ax = plt.subplots(figsize=(6, 6.5))
    for method, row in rows.iterrows():
        ax.scatter(
            row["mean_set_size"],
            row["singleton_rate"],
            s=100,
            color=METHOD_COLORS[method][PROBE],
            edgecolor="#222222",
            linewidth=0.75,
            alpha=0.92,
            label=PLOT_METHOD_LABELS.get(method, method),
            zorder=3,
        )

    x = rows["mean_set_size"]
    y = rows["singleton_rate"]
    x_padding = max(0.08, 0.12 * (x.max() - x.min()))
    y_padding = max(0.02, 0.30 * (y.max() - y.min()))
    ax.set_xlim(max(0, x.min() - x_padding), x.max() + x_padding)
    ax.set_ylim(max(0, y.min() - y_padding), min(1, y.max() + y_padding))
    ax.set_xlabel(r"Mean set size")
    ax.set_ylabel(r"Singleton rate (\%)")
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda value, _: f"{100 * value:.0f}")
    )
    ax.set_title(title)
    ax.set_axisbelow(True)
    ax.legend(
        ncol=1,
        frameon=False,
        loc="upper right",
        fontsize=14,
    )
    fig.tight_layout()

    if SAVE_FIGURES:
        fig.savefig(
            FIGURE_DIR / f"fig_conformal_{slug}_{PROBE}_{METHOD_COMPARISON_CP}_efficiency.pdf",
            bbox_inches="tight",
        )
    plt.show()
    return fig


efficiency_figures = {
    benchmark: plot_conformal_efficiency(benchmark)
    for benchmark in BENCHMARK_ORDER
}


## Reading the tables

- Marginal coverage is shown in a separate table against the nominal target, without best/second-best ranking: coverage is a validity criterion rather than a quantity to maximize. The class-conditional coverage gap (CovGap) is in notebook 10.
- Within each dataset and efficiency metric, the best displayed value is bold and the second-best is underlined. Mean set size is ranked from low to high, and singleton rate from high to low. Ties at the displayed precision receive the same emphasis.
- Mean set size and singleton rate are complementary efficiency measures. Smaller sets and more singleton predictions are desirable only while coverage remains adequate.
- HEST is the ID benchmark and is averaged across the 14 retained tissue sections. NuCLS, PanNuke, Lizard, PhenoCell (all), and PhenoCell (immune) are OOD benchmarks.
- The main table shades CellBridge (its two metric rows when displayed, its column in LaTeX).
